In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content
!rm -rf edge
!git clone https://github.com/stanford-tml/edge.git
%cd /content/edge

/content
Cloning into 'edge'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 52 (delta 4), reused 2 (delta 2), pack-reused 22 (from 1)
Receiving objects: 100% (52/52), 5.07 MiB | 33.07 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/edge


In [4]:
!pip install -q accelerate einops librosa soundfile tqdm matplotlib scipy numpy pandas p_tqdm wandb blobfile smplx gdown
!pip install -q git+https://github.com/rodrigo-castellon/jukemirlib.git
!pip install -q ninja fvcore iopath
!pip install -q "git+https://github.com/facebookresearch/pytorch3d.git"

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.2/82.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 9.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires dill<0.3.9,>=0.3.0, but you have dill 0.4.1 which is incompatible.
datasets 4.0.0 requires multiprocess<0.70.17, but you have multiprocess 0.70.19 which is incompatible.
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
!python -c "import pytorch3d; print('pytorch3d ok')"
!python -c "import jukemirlib; print('jukemirlib ok')"

pytorch3d ok
jukemirlib ok


In [6]:
!find /content/drive/MyDrive/EDGE_reproduction -name "checkpoint.pt" -type f

In [7]:
%cd /content/edge
!pip install -q -U gdown
!gdown "https://drive.google.com/uc?id=1BAR712cVEqB8GR37fcEihRV_xOC-fZrZ" -O checkpoint.pt
!ls -lh checkpoint.pt

/content/edge
Downloading...
From (original): https://drive.google.com/uc?id=1BAR712cVEqB8GR37fcEihRV_xOC-fZrZ
From (redirected): https://drive.google.com/uc?id=1BAR712cVEqB8GR37fcEihRV_xOC-fZrZ&confirm=t&uuid=21b890db-0bdb-4d36-84ad-012b2bc4a759
To: /content/edge/checkpoint.pt
100% 1.19G/1.19G [00:18<00:00, 64.1MB/s]
-rw------- 1 root root 1.2G Oct  3  2025 checkpoint.pt


In [8]:
!mkdir -p /content/drive/MyDrive/EDGE_reproduction/model
!cp checkpoint.pt /content/drive/MyDrive/EDGE_reproduction/model/checkpoint.pt
!ls -lh /content/drive/MyDrive/EDGE_reproduction/model/checkpoint.pt

-rw------- 1 root root 1.2G Apr 25 21:39 /content/drive/MyDrive/EDGE_reproduction/model/checkpoint.pt


In [9]:
from pathlib import Path

p = Path("/content/edge/EDGE.py")
text = p.read_text()

old = """checkpoint = torch.load(
                checkpoint_path, map_location=self.accelerator.device
            )"""

new = """checkpoint = torch.load(
                checkpoint_path, map_location=self.accelerator.device, weights_only=False
            )"""

if old in text:
    text = text.replace(old, new)
    p.write_text(text)
    print("patched EDGE.py")
else:
    print("Pattern not found or already patched")

patched EDGE.py


In [10]:
!python test.py --help

usage: test.py [-h] [--feature_type FEATURE_TYPE] [--out_length OUT_LENGTH]
               [--processed_data_dir PROCESSED_DATA_DIR]
               [--render_dir RENDER_DIR] [--checkpoint CHECKPOINT]
               [--music_dir MUSIC_DIR] [--save_motions]
               [--motion_save_dir MOTION_SAVE_DIR] [--cache_features]
               [--no_render] [--use_cached_features]
               [--feature_cache_dir FEATURE_CACHE_DIR]

options:
  -h, --help            show this help message and exit
  --feature_type FEATURE_TYPE
  --out_length OUT_LENGTH
                        max. length of output, in seconds
  --processed_data_dir PROCESSED_DATA_DIR
                        Dataset backup path
  --render_dir RENDER_DIR
                        Sample render path
  --checkpoint CHECKPOINT
                        checkpoint
  --music_dir MUSIC_DIR
                        folder containing input music
  --save_motions        Saves the motions for evaluation
  --motion_save_dir MOTION_SAVE_DIR

In [11]:
%cd /content/edge
!ls

/content/edge
args.py        dataset		  EDGE.py  media	README.md    train.py
checkpoint.pt  demo.ipynb	  eval	   model	SMPL-to-FBX  vis.py
data	       download_model.sh  LICENSE  __pycache__	test.py


In [12]:
!pip install -q openai librosa soundfile pandas numpy

In [13]:
%cd /content/edge
!mkdir -p full_music

/content/edge


In [14]:
!ffprobe -v error -show_entries format=duration \
  -of default=noprint_wrappers=1:nokey=1 full_music/full_song.mp3

228.109189


In [15]:
!mkdir -p full_music_clean

!ffmpeg -y -i full_music/full_song.mp3 \
  -t 228 \
  -ac 1 \
  -ar 44100 \
  -b:a 96k \
  full_music_clean/full_song.mp3

!ls -lh full_music_clean/full_song.mp3

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

# Introduce LLM

In [16]:
%%writefile llm_audio_segmenter.py
import json
import base64
import subprocess
from pathlib import Path


def audio_to_base64(audio_path):
    audio_path = Path(audio_path)
    return base64.b64encode(audio_path.read_bytes()).decode("utf-8")


def infer_audio_format(audio_path):
    suffix = Path(audio_path).suffix.lower().replace(".", "")
    if suffix not in ["wav", "mp3"]:
        raise ValueError(f"Unsupported audio format: {suffix}. Use wav or mp3.")
    return suffix


def get_audio_duration(audio_path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(audio_path)
    ]
    return float(subprocess.check_output(cmd).decode().strip())


def build_segmentation_prompt(audio_name, duration):
    return f"""
You are an audio-aware choreography director for a music-to-dance generation system.

Your task is to listen to the full song and divide it into a small number of meaningful dance-generation segments.
These segments will be cut into separate audio clips and passed one by one to the EDGE dance generation model.

Audio file: {audio_name}
Audio duration: {duration:.2f} seconds

Return valid JSON only with this schema:

{{
  "audio_file": "{audio_name}",
  "overall_mood": "short description",
  "danceability": 1-5,
  "beat_clarity": 1-5,
  "energy_level": 1-5,
  "segmentation_confidence": 1-5,
  "segments": [
    {{
      "segment_id": 0,
      "start_sec": 0.0,
      "end_sec": 30.0,
      "label": "intro | verse | chorus | bridge | outro | transition | unknown",
      "music_description": "brief description",
      "dance_goal": "brief dance instruction",
      "desired_motion_intensity": "low | medium | high | transitional",
      "selection_rule": "how to select or interpret EDGE output for this segment"
    }}
  ],
  "overall_warning": "brief warning about possible EDGE failure"
}}

Rules:
- Segment boundaries must be within 0 and {duration:.2f}.
- The first segment must start at 0.0.
- The last segment must end at {duration:.2f}.
- Use 4 to 7 segments.
- Avoid segments shorter than 15 seconds.
- Prefer segments between 20 and 45 seconds when possible.
- Do not create too many tiny segments.
- Use approximate labels. Do not overclaim exact verse/chorus boundaries.
- Make the segmentation useful for dance generation.
"""


def fallback_segment_plan(audio_path):
    audio_path = Path(audio_path)
    duration = get_audio_duration(audio_path)

    # robust fallback for long songs: 6 roughly equal segments
    n = 6 if duration >= 150 else 4
    boundaries = [duration * i / n for i in range(n + 1)]

    labels = ["intro", "verse", "chorus", "verse", "bridge", "outro"] if n == 6 else ["intro", "verse", "chorus", "outro"]

    segments = []
    for i in range(n):
        start = round(boundaries[i], 2)
        end = round(boundaries[i + 1], 2)
        label = labels[i]

        if label == "intro":
            intensity = "low"
            goal = "Open the choreography and establish the rhythm."
        elif label == "chorus":
            intensity = "high"
            goal = "Use larger and more energetic movement."
        elif label == "bridge":
            intensity = "transitional"
            goal = "Create contrast and transition into the ending."
        elif label == "outro":
            intensity = "low"
            goal = "Close the choreography smoothly."
        else:
            intensity = "medium"
            goal = "Maintain stable groove and moderate movement."

        segments.append({
            "segment_id": i,
            "start_sec": start,
            "end_sec": end,
            "label": label,
            "music_description": "Fallback segmentation; no audio LLM analysis was used.",
            "dance_goal": goal,
            "desired_motion_intensity": intensity,
            "selection_rule": "Generate with EDGE and later evaluate with PFC and visual inspection."
        })

    return {
        "audio_file": audio_path.name,
        "overall_mood": "unknown; fallback mode",
        "danceability": None,
        "beat_clarity": None,
        "energy_level": None,
        "segmentation_confidence": None,
        "segments": segments,
        "overall_warning": "Fallback segmentation only. This is not an audio LLM result."
    }


def call_audio_llm_segmenter(audio_path, model="gpt-4o-audio-preview"):
    from openai import OpenAI

    audio_path = Path(audio_path)
    duration = get_audio_duration(audio_path)
    audio_format = infer_audio_format(audio_path)
    audio_b64 = audio_to_base64(audio_path)

    prompt = build_segmentation_prompt(audio_path.name, duration)

    client = OpenAI()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "input_audio",
                        "input_audio": {
                            "data": audio_b64,
                            "format": audio_format
                        }
                    }
                ],
            }
        ],
    )

    text = response.choices[0].message.content

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {
            "raw_output": text,
            "parse_warning": "Model did not return valid JSON."
        }


def clean_and_validate_plan(plan, audio_path):
    audio_path = Path(audio_path)
    duration = get_audio_duration(audio_path)

    if "segments" not in plan or not isinstance(plan["segments"], list) or len(plan["segments"]) == 0:
        return fallback_segment_plan(audio_path)

    cleaned = []

    for seg in plan["segments"]:
        try:
            start = float(seg.get("start_sec", 0))
            end = float(seg.get("end_sec", duration))
        except Exception:
            continue

        start = max(0.0, min(start, duration))
        end = max(0.0, min(end, duration))

        if end <= start:
            continue

        cleaned.append({
            "segment_id": len(cleaned),
            "start_sec": round(start, 2),
            "end_sec": round(end, 2),
            "label": seg.get("label", "unknown"),
            "music_description": seg.get("music_description", ""),
            "dance_goal": seg.get("dance_goal", ""),
            "desired_motion_intensity": seg.get("desired_motion_intensity", "medium"),
            "selection_rule": seg.get("selection_rule", "")
        })

    if not cleaned:
        return fallback_segment_plan(audio_path)

    # Sort and force coverage.
    cleaned = sorted(cleaned, key=lambda x: x["start_sec"])
    cleaned[0]["start_sec"] = 0.0
    cleaned[-1]["end_sec"] = round(duration, 2)

    # Make adjacent boundaries continuous.
    for i in range(len(cleaned) - 1):
        midpoint = round((cleaned[i]["end_sec"] + cleaned[i + 1]["start_sec"]) / 2, 2)
        cleaned[i]["end_sec"] = midpoint
        cleaned[i + 1]["start_sec"] = midpoint

    # Re-number
    for i, seg in enumerate(cleaned):
        seg["segment_id"] = i

    plan["segments"] = cleaned
    plan["audio_file"] = audio_path.name
    return plan


def create_audio_segmentation_plan(
    audio_path,
    output_dir,
    use_llm=True,
    model="gpt-4o-audio-preview"
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    audio_path = Path(audio_path)
    base = audio_path.stem

    if use_llm:
        try:
            plan = call_audio_llm_segmenter(audio_path, model=model)
            source = "audio_llm"
        except Exception as e:
            plan = fallback_segment_plan(audio_path)
            source = f"fallback_after_audio_llm_error: {str(e)}"
    else:
        plan = fallback_segment_plan(audio_path)
        source = "fallback_no_llm"

    plan = clean_and_validate_plan(plan, audio_path)

    out = {
        "audio_file": audio_path.name,
        "plan_source": source,
        "model": model if use_llm else None,
        "plan": plan
    }

    out_path = output_dir / f"{base}_segmentation_plan.json"
    with open(out_path, "w") as f:
        json.dump(out, f, indent=2)

    return out_path

Writing llm_audio_segmenter.py


In [17]:
%%writefile edge_segmented_generation_per_segment.py
import json
import argparse
import subprocess
import shutil
from pathlib import Path

from llm_audio_segmenter import create_audio_segmentation_plan


def run_cmd(cmd):
    print("\n[CMD]", " ".join(str(x) for x in cmd))
    subprocess.run(cmd, check=True)


def safe_name(text):
    return "".join(c if c.isalnum() or c in ["_", "-"] else "_" for c in str(text))


def slice_one_segment(audio_path, seg, output_dir):
    audio_path = Path(audio_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    sid = int(seg["segment_id"])
    label = safe_name(seg.get("label", "segment"))
    start = float(seg["start_sec"])
    end = float(seg["end_sec"])
    duration = max(1.0, end - start)

    out_file = output_dir / f"{audio_path.stem}_seg{sid:02d}_{label}.wav"

    cmd = [
        "ffmpeg", "-y",
        "-ss", str(start),
        "-t", str(duration),
        "-i", str(audio_path),
        "-ac", "1",
        "-ar", "44100",
        str(out_file)
    ]
    run_cmd(cmd)

    return out_file, duration


def run_edge_for_one_segment(
    segment_audio_file,
    checkpoint,
    out_length,
    work_dir,
    segment_id,
    no_render=False
):
    segment_work = Path(work_dir) / f"edge_seg{segment_id:02d}"
    music_dir = segment_work / "music"
    motion_dir = segment_work / "motions"
    render_dir = segment_work / "renders"
    feature_cache_dir = segment_work / "feature_cache"

    music_dir.mkdir(parents=True, exist_ok=True)
    motion_dir.mkdir(parents=True, exist_ok=True)
    render_dir.mkdir(parents=True, exist_ok=True)
    feature_cache_dir.mkdir(parents=True, exist_ok=True)

    dst_audio = music_dir / Path(segment_audio_file).name
    shutil.copy(segment_audio_file, dst_audio)

    # EDGE sometimes behaves better with integer-ish out_length.
    # Keep one decimal to preserve approximate segment length.
    out_length = round(float(out_length), 1)

    cmd = [
        "python", "test.py",
        "--music_dir", str(music_dir),
        "--out_length", str(out_length),
        "--checkpoint", str(checkpoint),
        "--save_motions",
        "--motion_save_dir", str(motion_dir),
        "--render_dir", str(render_dir),
        "--feature_cache_dir", str(feature_cache_dir),
        "--cache_features"
    ]

    if no_render:
        cmd.append("--no_render")

    run_cmd(cmd)

    mp4s = sorted(render_dir.glob("*.mp4"))
    pkls = sorted(motion_dir.glob("*.pkl"))

    return {
        "segment_id": segment_id,
        "segment_work": str(segment_work),
        "music_dir": str(music_dir),
        "motion_dir": str(motion_dir),
        "render_dir": str(render_dir),
        "video_files": [str(x) for x in mp4s],
        "motion_files": [str(x) for x in pkls],
    }


def concatenate_videos(segment_results, output_video):
    video_files = []

    for res in segment_results:
        for v in res["video_files"]:
            video_files.append(Path(v))

    video_files = sorted(video_files)

    if not video_files:
        print("[Warning] No videos found to concatenate.")
        return None

    output_video = Path(output_video)
    output_video.parent.mkdir(parents=True, exist_ok=True)

    list_file = output_video.parent / "concat_list.txt"
    with open(list_file, "w") as f:
        for v in video_files:
            f.write(f"file '{v.resolve()}'\n")

    # Try stream copy first.
    cmd = [
        "ffmpeg", "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", str(list_file),
        "-c", "copy",
        str(output_video)
    ]

    try:
        run_cmd(cmd)
    except subprocess.CalledProcessError:
        cmd = [
            "ffmpeg", "-y",
            "-f", "concat",
            "-safe", "0",
            "-i", str(list_file),
            "-c:v", "libx264",
            "-c:a", "aac",
            str(output_video)
        ]
        run_cmd(cmd)

    return output_video


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--audio_path", type=str, required=True)
    parser.add_argument("--checkpoint", type=str, default="checkpoint.pt")
    parser.add_argument("--work_dir", type=str, default="segmented_edge_per_segment")

    parser.add_argument("--use_audio_llm", action="store_true")
    parser.add_argument("--audio_llm_model", type=str, default="gpt-4o-audio-preview")

    parser.add_argument("--no_render", action="store_true")
    parser.add_argument("--max_segments", type=int, default=7)

    args = parser.parse_args()

    audio_path = Path(args.audio_path)
    work_dir = Path(args.work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)

    plan_dir = work_dir / "plans"
    segment_audio_dir = work_dir / "segment_audio"
    edge_runs_dir = work_dir / "edge_runs"
    final_dir = work_dir / "final"

    plan_path = create_audio_segmentation_plan(
        audio_path=audio_path,
        output_dir=plan_dir,
        use_llm=args.use_audio_llm,
        model=args.audio_llm_model
    )

    print(f"[Segmented EDGE] Plan saved to {plan_path}")

    with open(plan_path) as f:
        plan_wrapper = json.load(f)

    segments = plan_wrapper["plan"]["segments"][: args.max_segments]

    sliced_metadata = []
    edge_results = []

    for seg in segments:
        sid = int(seg["segment_id"])
        print(f"\n========== Segment {sid}: {seg.get('label')} {seg['start_sec']}s-{seg['end_sec']}s ==========")

        segment_audio, duration = slice_one_segment(
            audio_path=audio_path,
            seg=seg,
            output_dir=segment_audio_dir
        )

        sliced_metadata.append({
            **seg,
            "segment_audio_file": str(segment_audio),
            "duration": duration
        })

        result = run_edge_for_one_segment(
            segment_audio_file=segment_audio,
            checkpoint=args.checkpoint,
            out_length=duration,
            work_dir=edge_runs_dir,
            segment_id=sid,
            no_render=args.no_render
        )
        edge_results.append(result)

    final_video = None
    if not args.no_render:
        final_video = concatenate_videos(
            edge_results,
            final_dir / f"{audio_path.stem}_llm_segmented_EDGE_full.mp4"
        )

    metadata = {
        "audio_path": str(audio_path),
        "plan_path": str(plan_path),
        "plan_source": plan_wrapper.get("plan_source"),
        "segments": sliced_metadata,
        "edge_results": edge_results,
        "final_video": str(final_video) if final_video else None,
        "interpretation": "The audio LLM controls segmentation; EDGE is invoked separately for each segment with matching out_length."
    }

    metadata_path = work_dir / "segmented_generation_metadata.json"
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"\n[Done] Metadata saved to {metadata_path}")
    if final_video:
        print(f"[Done] Final video saved to {final_video}")


if __name__ == "__main__":
    main()

Writing edge_segmented_generation_per_segment.py


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "......"

In [28]:
from openai import OpenAI

client = OpenAI()
print("API key loaded")

API key loaded


In [23]:
from jukemirlib.setup_models import setup_models

print("Downloading / loading Jukebox models...")
setup_models()
print("Jukebox models are ready.")

Downloading: 100% [10288727721 / 10288727721] bytesImporting jukebox and associated packages...
Setting up the VQ-VAE...
Loading vqvae in eval mode
Setting up the top prior...
Loading artist IDs from /usr/local/lib/python3.12/dist-packages/jukebox/data/ids/v2_artist_ids.txt
Loading artist IDs from /usr/local/lib/python3.12/dist-packages/jukebox/data/ids/v2_genre_ids.txt
Level:2, Cond downsample:None, Raw to tokens:128, Sample length:1048576
Converting to fp16 params
Loading prior in eval mode
Loading the top prior weights into memory...


100%|██████████| 669/669 [00:00<00:00, 50787.14it/s]

Jukebox models are ready.


In [24]:
!ls -lh full_music_clean

total 3.0M
-rw-r--r-- 1 root root 3.0M Apr 25 21:50 full_song.mp3


In [29]:
from openai import OpenAI

client = OpenAI()
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say OK only."}]
)

print(resp.choices[0].message.content)

OK


In [30]:
from openai import OpenAI
client = OpenAI()

print("API key works. Now you can rerun audio segmentation.")

API key works. Now you can rerun audio segmentation.


# Use LLM segement plan to generate dance

In [31]:
!python edge_segmented_generation_per_segment.py \
  --audio_path full_music_clean/full_song.mp3 \
  --checkpoint checkpoint.pt \
  --work_dir segmented_full_song_llm \
  --use_audio_llm \
  --audio_llm_model gpt-4o-audio-preview

[Segmented EDGE] Plan saved to segmented_full_song_llm/plans/full_song_segmentation_plan.json

========== Segment 0: intro 0.0s-30.0s ==========

[CMD] ffmpeg -y -ss 0.0 -t 30.0 -i full_music_clean/full_song.mp3 -ac 1 -ar 44100 segmented_full_song_llm/segment_audio/full_song_seg00_intro.wav
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-li

In [32]:
!cat segmented_full_song_llm/plans/full_song_segmentation_plan.json

{
  "audio_file": "full_song.mp3",
  "plan_source": "audio_llm",
  "model": "gpt-4o-audio-preview",
  "plan": {
    "audio_file": "full_song.mp3",
    "overall_mood": "Energetic, upbeat, and lively with a youthful, party vibe.",
    "danceability": 5,
    "beat_clarity": 4,
    "energy_level": 5,
    "segmentation_confidence": 4,
    "segments": [
      {
        "segment_id": 0,
        "start_sec": 0.0,
        "end_sec": 30.0,
        "label": "intro",
        "music_description": "Electronic dance intro with a slow build, smooth vocals, and steady beat",
        "dance_goal": "Start with soft, fluid movements to match the buildup and light beat drops.",
        "desired_motion_intensity": "low",
        "selection_rule": "Focus on graceful, introductory choreography for this segment."
      },
      {
        "segment_id": 1,
        "start_sec": 30.0,
        "end_sec": 65.0,
        "label": "verse",
        "music_description": "Vocals lead with a steady groove, rising energy.",

In [43]:
!find segmented_full_song_llm/edge_runs -type f -name "*.pkl" | sort

segmented_full_song_llm/edge_runs/edge_seg00/motions/test_full_song_seg00_intro.pkl
segmented_full_song_llm/edge_runs/edge_seg01/motions/test_full_song_seg01_verse.pkl
segmented_full_song_llm/edge_runs/edge_seg02/motions/test_full_song_seg02_chorus.pkl
segmented_full_song_llm/edge_runs/edge_seg03/motions/test_full_song_seg03_verse.pkl
segmented_full_song_llm/edge_runs/edge_seg04/motions/test_full_song_seg04_chorus.pkl
segmented_full_song_llm/edge_runs/edge_seg05/motions/test_full_song_seg05_bridge.pkl
segmented_full_song_llm/edge_runs/edge_seg06/motions/test_full_song_seg06_outro.pkl


In [44]:
import re
import csv
import shutil
import subprocess
from pathlib import Path
import pandas as pd

motion_files = sorted(Path("segmented_full_song_llm/edge_runs").glob("edge_seg*/motions/*.pkl"))

tmp_dir = Path("pfc_tmp_one_segment")
if tmp_dir.exists():
    shutil.rmtree(tmp_dir)
tmp_dir.mkdir(parents=True, exist_ok=True)

rows = []

for f in motion_files:
    # 清空临时文件夹
    for old in tmp_dir.glob("*"):
        old.unlink()

    shutil.copy(f, tmp_dir / f.name)

    output = subprocess.check_output(
        ["python", "eval/eval_pfc.py", "--motion_path", str(tmp_dir)],
        text=True
    )

    match = re.search(r"mean PFC of ([0-9.]+)", output)
    pfc = float(match.group(1)) if match else None

    seg_name = f.parts[-3]  # edge_seg00, edge_seg01, etc.

    rows.append({
        "condition": "LLM-segmented EDGE",
        "segment": seg_name,
        "motion_file": str(f),
        "pfc": pfc
    })

    print(seg_name, f.name, "PFC =", pfc)

seg_pfc_df = pd.DataFrame(rows)
seg_pfc_df.to_csv("pfc_llm_segmented_by_segment.csv", index=False)
seg_pfc_df

edge_seg00 test_full_song_seg00_intro.pkl PFC = 1.6200147394211688
edge_seg01 test_full_song_seg01_verse.pkl PFC = 0.6394959476677486
edge_seg02 test_full_song_seg02_chorus.pkl PFC = 0.7315225107660591
edge_seg03 test_full_song_seg03_verse.pkl PFC = 1.0975073846760344
edge_seg04 test_full_song_seg04_chorus.pkl PFC = 1.665698620913793
edge_seg05 test_full_song_seg05_bridge.pkl PFC = 0.7188782023874463
edge_seg06 test_full_song_seg06_outro.pkl PFC = 1.2828387042228842


,condition,segment,motion_file,pfc
0,LLM-segmented EDGE,edge_seg00,segmented_full_song_llm/edge_runs/edge_seg00/m...,1.620015
1,LLM-segmented EDGE,edge_seg01,segmented_full_song_llm/edge_runs/edge_seg01/m...,0.639496
2,LLM-segmented EDGE,edge_seg02,segmented_full_song_llm/edge_runs/edge_seg02/m...,0.731523
3,LLM-segmented EDGE,edge_seg03,segmented_full_song_llm/edge_runs/edge_seg03/m...,1.097507
4,LLM-segmented EDGE,edge_seg04,segmented_full_song_llm/edge_runs/edge_seg04/m...,1.665699
5,LLM-segmented EDGE,edge_seg05,segmented_full_song_llm/edge_runs/edge_seg05/m...,0.718878
6,LLM-segmented EDGE,edge_seg06,segmented_full_song_llm/edge_runs/edge_seg06/m...,1.282839


In [45]:
!rm -rf segmented_llm_all_motions
!mkdir -p segmented_llm_all_motions

!find segmented_full_song_llm/edge_runs -type f -name "*.pkl" -exec cp {} segmented_llm_all_motions/ \;

!find segmented_llm_all_motions -type f | sort

segmented_llm_all_motions/test_full_song_seg00_intro.pkl
segmented_llm_all_motions/test_full_song_seg01_verse.pkl
segmented_llm_all_motions/test_full_song_seg02_chorus.pkl
segmented_llm_all_motions/test_full_song_seg03_verse.pkl
segmented_llm_all_motions/test_full_song_seg04_chorus.pkl
segmented_llm_all_motions/test_full_song_seg05_bridge.pkl
segmented_llm_all_motions/test_full_song_seg06_outro.pkl


In [46]:
!python eval/eval_pfc.py --motion_path segmented_llm_all_motions | tee pfc_llm_segmented_overall.txt

100% 7/7 [00:00<00:00, 2876.75it/s]
segmented_llm_all_motions has a mean PFC of 1.1079937300078764


In [47]:
!cat pfc_llm_segmented_overall.txt

segmented_llm_all_motions has a mean PFC of 1.1079937300078764


In [48]:
!ffprobe -v error -show_entries format=duration \
  -of default=noprint_wrappers=1:nokey=1 full_music_clean/full_song.mp3

228.048980


In [51]:
!rm -rf full_music_baseline_wav
!mkdir -p full_music_baseline_wav

!ffmpeg -y -i full_music_clean/full_song.mp3 \
  -ac 1 \
  -ar 44100 \
  full_music_baseline_wav/full_song.wav

!ls -lh full_music_baseline_wav

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

# Use EDGE to generate the full song directly

In [52]:
!rm -rf full_song_baseline_motions full_song_baseline_renders full_song_baseline_cache
!mkdir -p full_song_baseline_motions full_song_baseline_renders full_song_baseline_cache

!python test.py \
  --music_dir full_music_baseline_wav \
  --out_length 228 \
  --checkpoint checkpoint.pt \
  --save_motions \
  --motion_save_dir full_song_baseline_motions \
  --render_dir full_song_baseline_renders \
  --feature_cache_dir full_song_baseline_cache \
  --cache_features \
  --no_render

Computing features for input music
Slicing full_music_baseline_wav/full_song.wav
Computing features for full_music_baseline_wav/full_song.wav
  0% 0/90 [00:00<?, ?it/s]Importing jukebox and associated packages...
Setting up the VQ-VAE...
Loading vqvae in eval mode
Setting up the top prior...
Loading artist IDs from /usr/local/lib/python3.12/dist-packages/jukebox/data/ids/v2_artist_ids.txt
Loading artist IDs from /usr/local/lib/python3.12/dist-packages/jukebox/data/ids/v2_genre_ids.txt
Level:2, Cond downsample:None, Raw to tokens:128, Sample length:1048576
Converting to fp16 params
Loading prior in eval mode
Loading the top prior weights into memory...

  0% 0/872 [00:00<?, ?it/s]
 22% 196/872 [00:00<00:00, 1955.23it/s]
 48% 416/872 [00:00<00:00, 2017.30it/s]
 73% 634/872 [00:00<00:00, 2060.85it/s]
100% 872/872 [00:00<00:00, 2112.22it/s]

100% 669/669 [00:00<00:00, 52399.43it/s]
100% 90/90 [01:49<00:00,  1.22s/it]
Model has 49464471 parameters
Generating dances
sampling loop time step: 

In [53]:
!find full_song_baseline_motions -type f -name "*.pkl" | sort

full_song_baseline_motions/test_full_song.pkl


In [54]:
!python eval/eval_pfc.py --motion_path full_song_baseline_motions | tee pfc_baseline_full_song.txt

100% 1/1 [00:00<00:00, 425.30it/s]
full_song_baseline_motions has a mean PFC of 0.39575002941486964


In [57]:
!find segmented_full_song_llm/edge_runs -type f -name "*.pkl" | sort
!find segmented_full_song_llm/segment_audio -type f -name "*.wav" | sort
!find full_song_baseline_motions -type f -name "*.pkl" | sort
!ls -lh full_music_baseline_wav

segmented_full_song_llm/edge_runs/edge_seg00/motions/test_full_song_seg00_intro.pkl
segmented_full_song_llm/edge_runs/edge_seg01/motions/test_full_song_seg01_verse.pkl
segmented_full_song_llm/edge_runs/edge_seg02/motions/test_full_song_seg02_chorus.pkl
segmented_full_song_llm/edge_runs/edge_seg03/motions/test_full_song_seg03_verse.pkl
segmented_full_song_llm/edge_runs/edge_seg04/motions/test_full_song_seg04_chorus.pkl
segmented_full_song_llm/edge_runs/edge_seg05/motions/test_full_song_seg05_bridge.pkl
segmented_full_song_llm/edge_runs/edge_seg06/motions/test_full_song_seg06_outro.pkl
segmented_full_song_llm/segment_audio/full_song_seg00_intro.wav
segmented_full_song_llm/segment_audio/full_song_seg01_verse.wav
segmented_full_song_llm/segment_audio/full_song_seg02_chorus.wav
segmented_full_song_llm/segment_audio/full_song_seg03_verse.wav
segmented_full_song_llm/segment_audio/full_song_seg04_chorus.wav
segmented_full_song_llm/segment_audio/full_song_seg05_bridge.wav
segmented_full_song_ll

# Beat Alignment

In [58]:
import pickle
import re
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
from scipy.signal import argrelextrema

FPS = 30
SIGMA_SECONDS = 0.10  # 越小越严格；0.10 秒约等于 3 帧


def load_motion_array(pkl_path):
    """
    Load EDGE saved motion .pkl and find the main numeric array.
    """
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    if isinstance(data, np.ndarray):
        arr = data

    elif isinstance(data, dict):
        preferred_keys = [
            "position", "positions", "pos",
            "joints", "joint_positions",
            "motion", "motions",
            "sample", "samples",
            "prediction", "pred"
        ]

        arr = None

        for key in preferred_keys:
            if key in data and isinstance(data[key], np.ndarray):
                arr = data[key]
                print(f"[Info] Using key '{key}' for {pkl_path}")
                break

        if arr is None:
            arrays = []
            for k, v in data.items():
                if isinstance(v, np.ndarray) and np.issubdtype(v.dtype, np.number):
                    arrays.append((k, v))

            if not arrays:
                raise ValueError(f"No numeric array found in {pkl_path}")

            arrays = sorted(arrays, key=lambda kv: kv[1].size, reverse=True)
            arr = arrays[0][1]
            print(f"[Info] Using largest numeric key '{arrays[0][0]}' for {pkl_path}")

    else:
        raise ValueError(f"Unsupported pkl type: {type(data)}")

    arr = np.asarray(arr, dtype=float)

    # Remove batch dimension if present
    if arr.ndim >= 4 and arr.shape[0] == 1:
        arr = arr[0]
    if arr.ndim >= 3 and arr.shape[0] == 1:
        arr = arr[0]

    return arr


def motion_energy_curve(pkl_path):
    """
    Frame-to-frame movement energy.
    """
    arr = load_motion_array(pkl_path)

    if arr.ndim < 2:
        raise ValueError(f"Motion array too small: {arr.shape}")

    flat = arr.reshape(arr.shape[0], -1)
    diff = np.diff(flat, axis=0)
    energy = np.linalg.norm(diff, axis=1)

    return energy


def detect_motion_beats(pkl_path, fps=30):
    """
    Following the common beat-alignment intuition:
    motion beats are local minima of kinetic velocity / motion energy.
    """
    energy = motion_energy_curve(pkl_path)

    if len(energy) < 10:
        return np.array([])

    minima = argrelextrema(energy, np.less, order=3)[0]

    if len(minima) == 0:
        return np.array([])

    motion_beat_times = minima / fps
    return motion_beat_times


def detect_music_beats(audio_path):
    """
    Detect music beats using librosa.
    """
    y, sr = librosa.load(audio_path, sr=None, mono=True)
    tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
    beat_times = librosa.frames_to_time(beat_frames, sr=sr)

    return beat_times, float(np.asarray(tempo).reshape(-1)[0])


def beat_alignment_score(audio_path, motion_path, sigma=SIGMA_SECONDS):
    """
    For each music beat, find the nearest motion beat.
    Higher score means better beat alignment.
    """
    music_beats, tempo = detect_music_beats(audio_path)
    motion_beats = detect_motion_beats(motion_path, fps=FPS)

    if len(music_beats) == 0 or len(motion_beats) == 0:
        return {
            "beat_alignment": 0.0,
            "tempo": tempo,
            "n_music_beats": len(music_beats),
            "n_motion_beats": len(motion_beats),
            "mean_nearest_distance_sec": None,
        }

    distances = []

    for mb in music_beats:
        nearest_distance = np.min(np.abs(motion_beats - mb))
        distances.append(nearest_distance)

    distances = np.array(distances)

    score = np.mean(np.exp(-(distances ** 2) / (2 * sigma ** 2)))

    return {
        "beat_alignment": float(score),
        "tempo": tempo,
        "n_music_beats": int(len(music_beats)),
        "n_motion_beats": int(len(motion_beats)),
        "mean_nearest_distance_sec": float(np.mean(distances)),
    }


def get_seg_id_from_name(path):
    """
    Extract segment id from names like:
    full_song_seg04_chorus.wav
    test_full_song_seg04_chorus.pkl
    """
    name = Path(path).stem
    m = re.search(r"seg(\d+)", name)
    return int(m.group(1)) if m else None

In [60]:
import scipy.signal

# Fix librosa compatibility with newer scipy
if not hasattr(scipy.signal, "hann"):
    scipy.signal.hann = scipy.signal.windows.hann

print("scipy.signal.hann patched:", hasattr(scipy.signal, "hann"))

scipy.signal.hann patched: True


In [61]:
segment_motion_files = sorted(Path("segmented_full_song_llm/edge_runs").glob("edge_seg*/motions/*.pkl"))
segment_audio_files = sorted(Path("segmented_full_song_llm/segment_audio").glob("*.wav"))

print("Segment motion files:", len(segment_motion_files))
for f in segment_motion_files:
    print("motion:", f)

print("\nSegment audio files:", len(segment_audio_files))
for f in segment_audio_files:
    print("audio:", f)

audio_by_seg = {get_seg_id_from_name(p): p for p in segment_audio_files}
motion_by_seg = {get_seg_id_from_name(p): p for p in segment_motion_files}

rows = []

for seg_id in sorted(set(audio_by_seg.keys()) & set(motion_by_seg.keys())):
    audio_path = audio_by_seg[seg_id]
    motion_path = motion_by_seg[seg_id]

    metrics = beat_alignment_score(audio_path, motion_path)

    rows.append({
        "condition": "LLM-segmented EDGE",
        "segment_id": seg_id,
        "audio_file": str(audio_path),
        "motion_file": str(motion_path),
        **metrics
    })

beat_llm_segment_df = pd.DataFrame(rows)
beat_llm_segment_df.to_csv("beat_alignment_llm_by_segment.csv", index=False)

beat_llm_segment_df

Segment motion files: 7
motion: segmented_full_song_llm/edge_runs/edge_seg00/motions/test_full_song_seg00_intro.pkl
motion: segmented_full_song_llm/edge_runs/edge_seg01/motions/test_full_song_seg01_verse.pkl
motion: segmented_full_song_llm/edge_runs/edge_seg02/motions/test_full_song_seg02_chorus.pkl
motion: segmented_full_song_llm/edge_runs/edge_seg03/motions/test_full_song_seg03_verse.pkl
motion: segmented_full_song_llm/edge_runs/edge_seg04/motions/test_full_song_seg04_chorus.pkl
motion: segmented_full_song_llm/edge_runs/edge_seg05/motions/test_full_song_seg05_bridge.pkl
motion: segmented_full_song_llm/edge_runs/edge_seg06/motions/test_full_song_seg06_outro.pkl

Segment audio files: 7
audio: segmented_full_song_llm/segment_audio/full_song_seg00_intro.wav
audio: segmented_full_song_llm/segment_audio/full_song_seg01_verse.wav
audio: segmented_full_song_llm/segment_audio/full_song_seg02_chorus.wav
audio: segmented_full_song_llm/segment_audio/full_song_seg03_verse.wav
audio: segmented_ful

,condition,segment_id,audio_file,motion_file,beat_alignment,tempo,n_music_beats,n_motion_beats,mean_nearest_distance_sec
0,LLM-segmented EDGE,0,segmented_full_song_llm/segment_audio/full_son...,segmented_full_song_llm/edge_runs/edge_seg00/m...,0.652526,129.199219,62,101,0.091467
1,LLM-segmented EDGE,1,segmented_full_song_llm/segment_audio/full_son...,segmented_full_song_llm/edge_runs/edge_seg01/m...,0.629869,129.199219,71,108,0.098107
2,LLM-segmented EDGE,2,segmented_full_song_llm/segment_audio/full_son...,segmented_full_song_llm/edge_runs/edge_seg02/m...,0.679861,129.199219,84,131,0.084611
3,LLM-segmented EDGE,3,segmented_full_song_llm/segment_audio/full_son...,segmented_full_song_llm/edge_runs/edge_seg03/m...,0.664515,129.199219,55,94,0.092593
4,LLM-segmented EDGE,4,segmented_full_song_llm/segment_audio/full_son...,segmented_full_song_llm/edge_runs/edge_seg04/m...,0.701158,129.199219,74,112,0.086393
5,LLM-segmented EDGE,5,segmented_full_song_llm/segment_audio/full_son...,segmented_full_song_llm/edge_runs/edge_seg05/m...,0.642397,129.199219,63,97,0.095805
6,LLM-segmented EDGE,6,segmented_full_song_llm/segment_audio/full_son...,segmented_full_song_llm/edge_runs/edge_seg06/m...,0.659653,129.199219,53,91,0.088418


In [62]:
llm_beat_alignment_mean = beat_llm_segment_df["beat_alignment"].mean()

llm_beat_alignment_summary = pd.DataFrame([{
    "condition": "LLM-segmented EDGE",
    "mean_beat_alignment": llm_beat_alignment_mean,
    "n_segments": len(beat_llm_segment_df),
    "total_music_beats": beat_llm_segment_df["n_music_beats"].sum(),
    "total_motion_beats": beat_llm_segment_df["n_motion_beats"].sum(),
    "mean_nearest_distance_sec": beat_llm_segment_df["mean_nearest_distance_sec"].mean()
}])

llm_beat_alignment_summary.to_csv("beat_alignment_llm_summary.csv", index=False)
llm_beat_alignment_summary

,condition,mean_beat_alignment,n_segments,total_music_beats,total_motion_beats,mean_nearest_distance_sec
0,LLM-segmented EDGE,0.661426,7,462,734,0.091056


In [63]:
baseline_audio = Path("full_music_baseline_wav/full_song.wav")
baseline_motion_files = sorted(Path("full_song_baseline_motions").glob("*.pkl"))

print("Baseline audio:", baseline_audio)
print("Baseline motions:", baseline_motion_files)

baseline_motion = baseline_motion_files[0]

baseline_beat_metrics = beat_alignment_score(baseline_audio, baseline_motion)

beat_baseline_df = pd.DataFrame([{
    "condition": "Baseline EDGE full-song",
    "audio_file": str(baseline_audio),
    "motion_file": str(baseline_motion),
    **baseline_beat_metrics
}])

beat_baseline_df.to_csv("beat_alignment_baseline_full_song.csv", index=False)
beat_baseline_df

Baseline audio: full_music_baseline_wav/full_song.wav
Baseline motions: [PosixPath('full_song_baseline_motions/test_full_song.pkl')]
[Info] Using largest numeric key 'smpl_poses' for full_song_baseline_motions/test_full_song.pkl


,condition,audio_file,motion_file,beat_alignment,tempo,n_music_beats,n_motion_beats,mean_nearest_distance_sec
0,Baseline EDGE full-song,full_music_baseline_wav/full_song.wav,full_song_baseline_motions/test_full_song.pkl,0.664758,129.199219,480,721,0.089737


In [64]:
beat_compare_df = pd.DataFrame([
    {
        "condition": "Baseline EDGE full-song",
        "beat_alignment": beat_baseline_df.loc[0, "beat_alignment"],
        "tempo": beat_baseline_df.loc[0, "tempo"],
        "n_music_beats": beat_baseline_df.loc[0, "n_music_beats"],
        "n_motion_beats": beat_baseline_df.loc[0, "n_motion_beats"],
        "mean_nearest_distance_sec": beat_baseline_df.loc[0, "mean_nearest_distance_sec"],
        "note": "Computed on the full song and full generated motion"
    },
    {
        "condition": "LLM-segmented EDGE",
        "beat_alignment": llm_beat_alignment_mean,
        "tempo": beat_llm_segment_df["tempo"].mean(),
        "n_music_beats": beat_llm_segment_df["n_music_beats"].sum(),
        "n_motion_beats": beat_llm_segment_df["n_motion_beats"].sum(),
        "mean_nearest_distance_sec": beat_llm_segment_df["mean_nearest_distance_sec"].mean(),
        "note": "Mean beat alignment across LLM-defined segments"
    }
])

beat_compare_df["change_vs_baseline"] = (
    beat_compare_df["beat_alignment"] - beat_compare_df.loc[0, "beat_alignment"]
)

beat_compare_df.to_csv("beat_alignment_comparison.csv", index=False)
beat_compare_df

,condition,beat_alignment,tempo,n_music_beats,n_motion_beats,mean_nearest_distance_sec,note,change_vs_baseline
0,Baseline EDGE full-song,0.664758,129.199219,480,721,0.089737,Computed on the full song and full generated m...,0.000000
1,LLM-segmented EDGE,0.661426,129.199219,462,734,0.091056,Mean beat alignment across LLM-defined segments,-0.003332


In [65]:
!mkdir -p /content/drive/MyDrive/EDGE_reproduction/segmented_generation/results

!cp beat_alignment_llm_by_segment.csv \
    beat_alignment_llm_summary.csv \
    beat_alignment_baseline_full_song.csv \
    beat_alignment_comparison.csv \
    /content/drive/MyDrive/EDGE_reproduction/segmented_generation/results/

# Within-output diversity proxy

In [66]:
from scipy.spatial.distance import pdist

CHUNK_SECONDS = 5
CHUNK_FRAMES = FPS * CHUNK_SECONDS


def get_chunks_from_motion(pkl_path, chunk_frames=CHUNK_FRAMES):
    arr = load_motion_array(pkl_path)

    if arr.ndim < 2:
        raise ValueError(f"Motion array too small: {arr.shape}")

    flat = arr.reshape(arr.shape[0], -1)
    chunks = []

    n = flat.shape[0]
    for start in range(0, n - chunk_frames + 1, chunk_frames):
        end = start + chunk_frames
        chunks.append(flat[start:end])

    return chunks


def kinetic_feature(chunk):
    diff = np.diff(chunk, axis=0)
    speed = np.linalg.norm(diff, axis=1)

    accel = np.diff(diff, axis=0)
    accel_norm = np.linalg.norm(accel, axis=1)

    return np.array([
        np.mean(speed),
        np.std(speed),
        np.max(speed),
        np.percentile(speed, 25),
        np.percentile(speed, 50),
        np.percentile(speed, 75),
        np.mean(accel_norm),
        np.std(accel_norm),
        np.max(accel_norm),
    ], dtype=float)


def geometric_feature(chunk):
    centered = chunk - chunk.mean(axis=1, keepdims=True)

    mean_pose = centered.mean(axis=0)
    std_pose = centered.std(axis=0)

    return np.array([
        np.mean(mean_pose),
        np.std(mean_pose),
        np.mean(std_pose),
        np.std(std_pose),
        np.percentile(centered, 10),
        np.percentile(centered, 25),
        np.percentile(centered, 50),
        np.percentile(centered, 75),
        np.percentile(centered, 90),
    ], dtype=float)


def collect_features_from_motion_files(motion_files):
    kinetic_features = []
    geometric_features = []
    chunk_sources = []

    for f in motion_files:
        chunks = get_chunks_from_motion(f)

        for i, chunk in enumerate(chunks):
            kinetic_features.append(kinetic_feature(chunk))
            geometric_features.append(geometric_feature(chunk))
            chunk_sources.append({
                "motion_file": str(f),
                "chunk_id": i
            })

    return np.vstack(kinetic_features), np.vstack(geometric_features), chunk_sources


def mean_pairwise_distance(features):
    if len(features) < 2:
        return np.nan
    return float(np.mean(pdist(features, metric="euclidean")))

In [67]:
baseline_motion_files = sorted(Path("full_song_baseline_motions").glob("*.pkl"))
llm_motion_files = sorted(Path("segmented_full_song_llm/edge_runs").glob("edge_seg*/motions/*.pkl"))

base_k, base_g, base_chunks = collect_features_from_motion_files(baseline_motion_files)
llm_k, llm_g, llm_chunks = collect_features_from_motion_files(llm_motion_files)

def standardize_two_groups(a, b):
    all_feat = np.vstack([a, b])
    mean = all_feat.mean(axis=0, keepdims=True)
    std = all_feat.std(axis=0, keepdims=True) + 1e-8
    return (a - mean) / std, (b - mean) / std

base_k_z, llm_k_z = standardize_two_groups(base_k, llm_k)
base_g_z, llm_g_z = standardize_two_groups(base_g, llm_g)

diversity_df = pd.DataFrame([
    {
        "condition": "Baseline EDGE full-song",
        "n_5sec_chunks": len(base_k),
        "Dist_k_proxy": mean_pairwise_distance(base_k_z),
        "Dist_g_proxy": mean_pairwise_distance(base_g_z),
    },
    {
        "condition": "LLM-segmented EDGE",
        "n_5sec_chunks": len(llm_k),
        "Dist_k_proxy": mean_pairwise_distance(llm_k_z),
        "Dist_g_proxy": mean_pairwise_distance(llm_g_z),
    }
])

diversity_df["Dist_k_change_vs_baseline"] = (
    diversity_df["Dist_k_proxy"] - diversity_df.loc[0, "Dist_k_proxy"]
)
diversity_df["Dist_g_change_vs_baseline"] = (
    diversity_df["Dist_g_proxy"] - diversity_df.loc[0, "Dist_g_proxy"]
)

diversity_df.to_csv("diversity_proxy_comparison.csv", index=False)
diversity_df

[Info] Using largest numeric key 'smpl_poses' for full_song_baseline_motions/test_full_song.pkl
[Info] Using largest numeric key 'smpl_poses' for segmented_full_song_llm/edge_runs/edge_seg00/motions/test_full_song_seg00_intro.pkl
[Info] Using largest numeric key 'smpl_poses' for segmented_full_song_llm/edge_runs/edge_seg01/motions/test_full_song_seg01_verse.pkl
[Info] Using largest numeric key 'smpl_poses' for segmented_full_song_llm/edge_runs/edge_seg02/motions/test_full_song_seg02_chorus.pkl
[Info] Using largest numeric key 'smpl_poses' for segmented_full_song_llm/edge_runs/edge_seg03/motions/test_full_song_seg03_verse.pkl
[Info] Using largest numeric key 'smpl_poses' for segmented_full_song_llm/edge_runs/edge_seg04/motions/test_full_song_seg04_chorus.pkl
[Info] Using largest numeric key 'smpl_poses' for segmented_full_song_llm/edge_runs/edge_seg05/motions/test_full_song_seg05_bridge.pkl
[Info] Using largest numeric key 'smpl_poses' for segmented_full_song_llm/edge_runs/edge_seg06/mo

,condition,n_5sec_chunks,Dist_k_proxy,Dist_g_proxy,Dist_k_change_vs_baseline,Dist_g_change_vs_baseline
0,Baseline EDGE full-song,45,3.359733,4.089411,0.000000,0.000000
1,LLM-segmented EDGE,45,3.639452,3.523860,0.279719,-0.565551


In [68]:
!ls -lh pfc_baseline_full_song.txt pfc_llm_segmented_overall.txt

-rw-r--r-- 1 root root 65 Apr 25 23:35 pfc_baseline_full_song.txt
-rw-r--r-- 1 root root 63 Apr 25 23:30 pfc_llm_segmented_overall.txt


In [69]:
import re
from pathlib import Path

def extract_pfc(txt_path):
    text = Path(txt_path).read_text()
    match = re.search(r"mean PFC of ([0-9.]+)", text)
    return float(match.group(1)) if match else None

baseline_pfc = extract_pfc("pfc_baseline_full_song.txt")
llm_pfc = extract_pfc("pfc_llm_segmented_overall.txt")

final_metrics_df = pd.DataFrame([
    {
        "condition": "Baseline EDGE full-song",
        "PFC": baseline_pfc,
        "BeatAlignment": beat_baseline_df.loc[0, "beat_alignment"],
        "Dist_k_proxy": diversity_df.loc[0, "Dist_k_proxy"],
        "Dist_g_proxy": diversity_df.loc[0, "Dist_g_proxy"],
    },
    {
        "condition": "LLM-segmented EDGE",
        "PFC": llm_pfc,
        "BeatAlignment": llm_beat_alignment_mean,
        "Dist_k_proxy": diversity_df.loc[1, "Dist_k_proxy"],
        "Dist_g_proxy": diversity_df.loc[1, "Dist_g_proxy"],
    }
])

final_metrics_df["PFC_change_vs_baseline"] = (
    final_metrics_df["PFC"] - final_metrics_df.loc[0, "PFC"]
)
final_metrics_df["BeatAlignment_change_vs_baseline"] = (
    final_metrics_df["BeatAlignment"] - final_metrics_df.loc[0, "BeatAlignment"]
)
final_metrics_df["Dist_k_change_vs_baseline"] = (
    final_metrics_df["Dist_k_proxy"] - final_metrics_df.loc[0, "Dist_k_proxy"]
)
final_metrics_df["Dist_g_change_vs_baseline"] = (
    final_metrics_df["Dist_g_proxy"] - final_metrics_df.loc[0, "Dist_g_proxy"]
)

final_metrics_df.to_csv("final_metrics_pfc_beat_diversity.csv", index=False)
final_metrics_df

,condition,PFC,BeatAlignment,Dist_k_proxy,Dist_g_proxy,PFC_change_vs_baseline,BeatAlignment_change_vs_baseline,Dist_k_change_vs_baseline,Dist_g_change_vs_baseline
0,Baseline EDGE full-song,0.395750,0.664758,3.359733,4.089411,0.000000,0.000000,0.000000,0.000000
1,LLM-segmented EDGE,1.107994,0.661426,3.639452,3.523860,0.712244,-0.003332,0.279719,-0.565551


In [70]:
!cp diversity_proxy_comparison.csv \
    final_metrics_pfc_beat_diversity.csv \
    /content/drive/MyDrive/EDGE_reproduction/segmented_generation/results/

In [71]:
!grep -R "guidance" -n .
!grep -R "guidance_weight" -n .
!grep -R "cond_scale" -n .
!grep -R "w =" -n EDGE.py model test.py

./EDGE.py:90:            guidance_weight=2,
./model/diffusion.py:55:        guidance_weight=3,
./model/diffusion.py:86:        self.guidance_weight = guidance_weight
./model/diffusion.py:158:        weight = weight if weight is not None else self.guidance_weight
./model/diffusion.py:180:        # guidance clipping
./model/diffusion.py:182:            weight = min(self.guidance_weight, 0)
./model/diffusion.py:184:            weight = min(self.guidance_weight, 1)
./model/diffusion.py:186:            weight = self.guidance_weight
./model/diffusion.py:293:        weights = np.clip(np.linspace(0, self.guidance_weight * 2, sampling_timesteps), None, self.guidance_weight)
grep: ./model/__pycache__/diffusion.cpython-312.pyc: binary file matches
grep: ./model/__pycache__/model.cpython-312.pyc: binary file matches
./model/model.py:283:        # null embeddings for guidance dropout
./model/model.py:331:    def guided_forward(self, x, cond_embed, times, guidance_weight):
./model/model.py:335:     